# OpenPOPCON ITER example

The ITER baseline inductive scenario: 15 MA, 5.3 T, 500 MW of fusion power at
Q = 10. Machine parameters come from the ITER Physics Basis and Progress in
the ITER Physics Basis (references [7] in the repository README).

This example uses parabolic profiles rather than a gEQDSK, which keeps `R`,
`a`, `kappa`, `delta` and `I_P` free for `POPCON_scan` to vary.

In [ ]:
import numpy as np
import openpopcon as op

## Setup and run

`settingsfile` holds the machine and algorithm settings, `plotsettingsfile`
the contour levels and axes. Both are read from this directory.

The numerics are compiled with numba the first time they run, so the first
solve in a fresh kernel takes a few seconds longer than the rest.

In [ ]:
settingsfile = "./POPCON_input_example.yaml"
plotsettingsfile = "./plotsettings.yml"

pc = op.POPCON(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
pc.run_POPCON()

## Plotting

In [ ]:
fig, ax = pc.plot()

### Checking against the design point

The baseline sits near 0.85 of the Greenwald density at a volume-averaged ion
temperature around 8.5 keV. The fusion power there should come out close to
the nominal 500 MW.

Q comes out higher than the nominal 10. That is expected rather than wrong:
ITER's Q = 10 point carries 50 MW of external heating for burn control, not
because 50 MW is the minimum required, and a 0-D balance with parabolic
profiles finds the ignition boundary nearby. Read the Q contours as
optimistic.

In [ ]:
valid = pc.output.Paux < 99998.0
i = int(np.abs(pc.output.n_G_frac.values - 0.85).argmin())
j = int(np.abs(pc.output.T_i_avg.values - 8.5).argmin())
point = pc.output.isel(n_index=i, T_index=j)

print(f"n/n_G   = {float(point.n_G_frac):.2f}")
print(f"<T_i>   = {float(point.T_i_avg):.1f} keV")
print(f"P_fus   = {float(point.Pfusion):.0f} MW   (nominal 500)")
print(f"P_aux   = {float(point.Paux):.0f} MW")
print(f"Q       = {float(point.Q):.1f}")
print(f"beta_N  = {float(point.betaN):.2f}")

## Scoping a single operating point

`single_point` solves one density/temperature pair and shows the profiles
behind it, which is the quickest way to see why a point on the POPCON sits
where it does.

In [ ]:
pc.single_point(n_G_frac=0.85, Ti_av=8.5)

## Plotting against different axes

Either axis takes any density or temperature label, so the same solve can be
drawn with the density on x. The seven names are listed in
`openpopcon.PLOT_AXES` and in the plotsettings file.

In [ ]:
print(sorted(op.PLOT_AXES))

pc.plotsettings.xax = "nG"
pc.plotsettings.yax = "T_i_av"
fig, ax = pc.plot()

pc.plotsettings.xax = "T_i_av"
pc.plotsettings.yax = "nG"

## Scanning the current and the confinement factor

`POPCON_scan` runs a full POPCON at every combination of two machine
parameters. The ranges below come from the `scan:` block at the bottom of the
settings file; passing `scan=` to the constructor overrides it.

In [ ]:
sc = op.POPCON_scan(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
sc.run_scan()

In [ ]:
fig, axs = sc.plot()

`plot_metric` reduces each cell to one number so the trend across the scan
reads at a glance. Anything in the output works, with any reduction.

In [ ]:
fig, ax = sc.plot_metric("Pfusion", reduce="max")

The whole scan is also available as one xarray Dataset, with the two scanned
parameters as extra dimensions, for any analysis the plots do not cover.

In [ ]:
sc.output